# **Атаки на табличные объяснения: фон SHAP, PDP, контрфактические объяснения и fairwashing**

Практика к модулю [«Атаки на объяснения»](https://ai-interpretability.school), уроки «Манипуляция контрфактическими объяснениями и recourse», «Fairwashing: рационализация несправедливой модели» и «Атаки на background SHAP и на глобальные объяснения». Модель-эшафот Slack и соавторов (2020) разобрана в отдельной тетради `PRACTICE_fooling_lime_shap` и здесь не повторяется.

Все восемь атак считаем на одних данных — открытом наборе UCI Adult (перепись США 1994 года), — чтобы их можно было сравнить между собой:

| раздел | атака | что меняет противник | урок |
|---|---|---|---|
| 1 | Fool SHAP (Laberge и др., 2023) | веса строк фона, по которому считают SHAP | атаки на background SHAP |
| 2 | Fooling PD (Baniecki и др., 2023) | данные, по которым считают кривую частичной зависимости | атаки на background SHAP |
| 3 | Manipulated CFE (Slack и др., 2021), упрощенная версия | модель: у нее есть лазейка для поиска контрфактических объяснений | манипуляция контрфактическими объяснениями |
| 4 | CFD (Pawelczyk и др., 2023) | ничего: атакующая по контрфактическому объяснению узнает членство в обучающей выборке | манипуляция контрфактическими объяснениями |
| 5 | LaundryML (Aïvodji и др., 2019) | объяснение: из множества суррогатов выбирают справедливо выглядящий | fairwashing |
| 6 | ε-constrained fairwashing (Aïvodji и др., 2021) | объяснение: суррогат с ограничением на несправедливость | fairwashing |
| 7 | Off-manifold manipulation и tsp (Anders и др., 2020) | модель вне многообразия данных; защита — проекция на касательное пространство | fairwashing |
| 8 | Explanation loss (Dimanov и др., 2020) | модель: дообучение со штрафом на градиент по признаку | fairwashing |

В каждом разделе три вопроса: работает ли атака на этих данных, от какого гиперпараметра зависит ее сила и где она ломается. В конце — задания без ключей для самопроверки.

Тетрадь считается на CPU за три-четыре минуты.

In [ ]:
!pip install shap==0.52.0 dice-ml==0.12 fairlearn==0.14.0 -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # numpy, sklearn and tqdm print harmless warnings on some platforms
import copy, itertools, math, io, contextlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import shap
import dice_ml
from scipy import sparse, stats
from scipy.optimize import linprog
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import partial_dependence
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from fairlearn.reductions import ExponentiatedGradient, DemographicParity

SEED = 0
rng = np.random.default_rng(SEED)   # every numpy random step of the notebook draws from this generator
torch.manual_seed(SEED)
torch.set_num_threads(4)            # the same numbers on a laptop and in Colab

## Данные UCI Adult

Обучающая часть набора Adult из UCI Machine Learning Repository (Becker и Kohavi, 1996, лицензия CC BY 4.0, DOI [10.24432/C5XW20](https://doi.org/10.24432/C5XW20)), как в `PRACTICE_fooling_lime_shap`: 32561 человек из переписи США 1994 года. Целевая переменная — годовой доход выше 50 тысяч долларов, это благоприятный исход. Признаки — возраст, число лет образования (`education_num`), часы работы в неделю (`hours_per_week`), брак (`married`: 1 при семейном положении Married-civ-spouse или Married-AF-spouse), пол и раса. Чувствительный признак — `race_black`, группа $s=1$: люди, записанные в переписи с расой Black.

Непрерывные признаки стандартизуем, бинарные оставляем нулями и единицами. Данные делим на три части: обучение черного ящика (60%), suing group — люди, оспаривающие решения, на которых компания строит объяснение (20%), и отложенная выборка аудиторки (20%).

In [ ]:
URL = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main/data/adult.csv'
df = pd.read_csv(URL)
raw = pd.DataFrame({
    'age': df.age,
    'education_num': df['education-num'],
    'hours_per_week': df['hours-per-week'],
    'married': df['marital-status'].isin(['Married-civ-spouse', 'Married-AF-spouse']).astype(int),
    'sex_male': (df.sex == 'Male').astype(int),
    'race_black': (df.race == 'Black').astype(int),
})
y = (df.income == '>50K').astype(int).values
features = list(raw.columns)
CONT = ['age', 'education_num', 'hours_per_week']
RACE, AGE = features.index('race_black'), features.index('age')
mu, sd = raw[CONT].mean(), raw[CONT].std()
X = raw.astype(float)
X[CONT] = (X[CONT] - mu) / sd
X = X.values
s = X[:, RACE].astype(int)
idx_tr, idx_rest = train_test_split(np.arange(len(X)), test_size=0.4, random_state=SEED, stratify=y)
idx_sg, idx_ho = train_test_split(idx_rest, test_size=0.5, random_state=SEED, stratify=y[idx_rest])
print(len(X), 'объектов; обучение / suing group / отложенная:', len(idx_tr), '/', len(idx_sg), '/', len(idx_ho))
print('доля s=1:', round(s.mean(), 3))

Черный ящик — градиентный бустинг. Его несправедливость меряем разрывом демографического паритета: разностью долей прогноза «доход выше 50K» в группах $s=1$ и $s=0$.

In [ ]:
bb = HistGradientBoostingClassifier(max_iter=200, random_state=SEED).fit(X[idx_tr], y[idx_tr])
f_bb = lambda Z: bb.predict_proba(Z)[:, 1]

def dp_gap(pred, group):
    """Demographic parity gap |P(pred=1 | s=1) - P(pred=1 | s=0)| of 0/1 predictions or probabilities."""
    return abs(pred[group == 1].mean() - pred[group == 0].mean())

print('точность черного ящика на отложенной выборке:', round((bb.predict(X[idx_ho]) == y[idx_ho]).mean(), 3))
print('разрыв демографического паритета прогнозов на suing group:', round(dp_gap(bb.predict(X[idx_sg]), s[idx_sg]), 3))

## 1. Fool SHAP: веса фона, которые не ловят тесты

Сценарий аудита из урока «Атаки на background SHAP и на глобальные объяснения». Аудиторка проверяет демографический паритет и спрашивает, какая доля разрыва приходится на расу. Передний набор $\mathcal F$ берется из группы $s=0$, фон $\mathcal B$ — из группы $s=1$, и тогда сумма глобальных значений Шепли равна $\mathbb E[f\mid s=0]-\mathbb E[f\mid s=1]$, а значение расы $\Phi_s$ показывает ее долю в разрыве.

Глобальное значение — среднее по фону: $\Phi_s=\frac1{|\mathcal B|}\sum_{z\in\mathcal B}\Phi_s(z)$, где $\Phi_s(z)$ — значение расы с одной фоновой точкой $z$, усредненное по переднему набору. Поэтому, если строки фона $D_1$ брать с весами $\omega$, то $\Phi_s(\omega)=\sum_j\omega_j\Phi_s(z^{(j)})$ — линейная функция весов (Prop. 4.1 статьи). Сначала посчитаем значения всех признаков с одной фоновой точкой $z$ для каждой строки $D_1$ точно — перебором всех 64 коалиций шести признаков, — и сверим средние с `shap.ExactExplainer`.

In [ ]:
D0, D1 = X[s == 0], X[s == 1]
S0 = D0[rng.choice(len(D0), 100, replace=False)]   # unbiased foreground S'_0, 100 rows of group s=0
n_feat = X.shape[1]
coalitions = [S for k in range(n_feat + 1) for S in itertools.combinations(range(n_feat), k)]

def shapley_per_baseline(F, B):
    """Exact interventional Shapley values of every feature with a single baseline z, averaged over the foreground F.
    Returns an array (len(B), n_feat): one row per baseline row z of B."""
    Fx = np.tile(F, (len(B), 1))
    v = {}
    for S in coalitions:                   # v(S): mean prediction with features S from x and the rest from z
        H = np.repeat(B, len(F), axis=0)
        H[:, list(S)] = Fx[:, list(S)]
        v[S] = f_bb(H).reshape(len(B), len(F)).mean(axis=1)
    phi = np.zeros((len(B), n_feat))
    for S in coalitions:
        for i in range(n_feat):
            if i not in S:
                w = math.factorial(len(S)) * math.factorial(n_feat - len(S) - 1) / math.factorial(n_feat)
                phi[:, i] += w * (v[tuple(sorted(S + (i,)))] - v[S])
    return phi

Phi_z = shapley_per_baseline(S0, D1)          # one row per row z of D1
phi_j = Phi_z[:, RACE]                        # the value of race for every background row
B_unbiased = D1[rng.choice(len(D1), 200, replace=False)]
ex = shap.ExactExplainer(f_bb, shap.maskers.Independent(B_unbiased, max_samples=200))
sv_unbiased = ex(S0, silent=True).values
print('глобальные значения по shap.ExactExplainer, фон 200 случайных строк D1:', np.round(sv_unbiased.mean(0), 4))
print('то же из таблицы Phi_z:', np.round(shapley_per_baseline(S0, B_unbiased).mean(0), 4))
print('Phi_s по всей D1 (несмещенный фон):', round(phi_j.mean(), 4))
print('сумма глобальных значений и разность средних прогнозов групп:', round(Phi_z.mean(0).sum(), 4), round(f_bb(S0).mean() - f_bb(D1).mean(), 4))

Теперь подбор весов. Противник минимизирует $\beta\,\Phi_s(\omega)+\lambda\,\mathcal W(\mathcal B,\mathcal B'_\omega)$, где $\beta$ — знак $\Phi_s$ по несмещенному фону (алгоритм 1 статьи: первое слагаемое линейно, без модуля, поэтому при малом $\lambda$ вклад может перейти через ноль), а $\mathcal W$ — расстояние Вассерштейна между исходным и перевзвешенным фоном **в пространстве прогнозов модели**: именно распределение прогнозов проверяют тесты аудиторки. Для одномерных распределений $\mathcal W$ равно площади между функциями распределения, $\sum_k|F_\omega(p_k)-F(p_k)|\,(p_{k+1}-p_k)$ по отсортированным прогнозам $p_k$, и вся задача — линейная программа. В статье та же задача сведена к потоку минимальной стоимости; ответ тот же, решатель другой.

Аудиторка получает передний набор и выборку фона размера $M=200$, насэмплированную по весам $\omega$, и проверяет их, как в алгоритме 2 статьи: тестом Колмогорова–Смирнова сравнивает прогнозы на каждом подмножестве с прогнозами на своей случайной подвыборке того же размера, а тестом Вальда — со средним прогнозом всей группы. Тестов четыре, поэтому порог p-значения — $\alpha/4$ при $\alpha=0.05$ (поправка Бонферрони). Как в алгоритме 3, для каждого $\lambda$ считаем долю обнаружений по 100 выборкам фона; атаку там принимают, если эта доля ниже порога $\tau=10\%$. Передний набор у нас из 100 строк, а в статье оба подмножества размера $M$: таблица $\Phi_s(z)$ для всех 3124 строк $D_1$ считается перебором коалиций, и с меньшим передним набором она считается вдвое быстрее.

In [ ]:
pred1, pred0 = f_bb(D1), f_bb(D0)   # the auditor has the predictions on all of D1 and D0
order = np.argsort(pred1)
gaps = np.diff(pred1[order])
N1 = len(D1)
beta = np.sign(phi_j.mean())         # Algorithm 1 of the paper: the sign of Phi_s on the unbiased background

def fool_weights(lam):
    """Algorithm 1 of the paper: weights on the rows of D1 minimizing beta * sum w*phi + lam * W1(predictions of D1, predictions reweighted by w). A sparse LP."""
    # variables: w (N1, sorted by prediction), c (N1-1 cumulative sums of w), t (N1-1, |c_k - k/N1|)
    nv = N1 + 2 * (N1 - 1)
    cost = np.r_[beta * phi_j[order], np.zeros(N1 - 1), lam * gaps]
    rows, cols, vals = [], [], []
    for k in range(N1 - 1):                     # c_k - c_{k-1} - w_k = 0
        rows += [k, k]; cols += [N1 + k, k]; vals += [1.0, -1.0]
        if k > 0:
            rows.append(k); cols.append(N1 + k - 1); vals.append(-1.0)
    A_eq = sparse.vstack([sparse.csr_matrix((vals, (rows, cols)), shape=(N1 - 1, nv)),
                          sparse.csr_matrix((np.ones(N1), (np.zeros(N1), np.arange(N1))), shape=(1, nv))])
    b_eq = np.r_[np.zeros(N1 - 1), 1.0]
    ref = np.arange(1, N1) / N1
    I, Z = sparse.identity(N1 - 1), sparse.csr_matrix((N1 - 1, N1))
    A_ub = sparse.vstack([sparse.hstack([Z, I, -I]), sparse.hstack([Z, -I, -I])])
    b_ub = np.r_[ref, -ref]
    res = linprog(cost, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=(0, None), method='highs')
    w = np.zeros(N1)
    w[order] = res.x[:N1]
    return w / w.sum()

M, ALPHA, REPS = 200, 0.05, 100

def detect(S0_, S1_, r):
    """Algorithm 2 of the paper for both transmitted subsets: KS against the auditor's own uniform subsample
    of the same size and Wald against all predictions of the group, Bonferroni threshold ALPHA/4.
    Returns 1 if fraud is detected and the four p-values."""
    pv = []
    for S_, pred_all in [(S0_, pred0), (S1_, pred1)]:
        p = f_bb(S_)
        own = r.choice(pred_all, len(p), replace=False)
        pv += [stats.ks_2samp(own, p).pvalue,
               2 * stats.norm.sf(abs(p.mean() - pred_all.mean()) / (pred_all.std() / np.sqrt(len(p))))]
    return int(min(pv) < ALPHA / 4), np.round(pv, 3)

def detection_rate(w):
    """Algorithm 3 of the paper: share of REPS background samples of size M, drawn with weights w, that the auditor flags."""
    r = np.random.default_rng(SEED)
    return np.mean([detect(S0, D1[r.choice(N1, M, p=w)], r)[0] for _ in range(REPS)])

print(f'{"lambda":>7}{"Phi_s(w)":>11}{"сумма":>9}{"W1":>8}{"строк с весом":>16}{"обнаружение":>14}')
print(f'{"без весов":>7}{phi_j.mean():>11.4f}{Phi_z.mean(0).sum():>9.4f}{0:>8.4f}{N1:>16}{detection_rate(np.full(N1, 1 / N1)):>14.2f}')
weights = {}
for lam in [1, 3, 10, 30]:
    w = fool_weights(lam)
    weights[lam] = w
    W1 = np.abs(np.cumsum(w[order])[:-1] - np.arange(1, N1) / N1) @ gaps
    print(f'{lam:>7}{w @ phi_j:>11.4f}{(w @ Phi_z).sum():>9.4f}{W1:>8.4f}{(w > 1e-9).sum():>16}{detection_rate(w):>14.2f}')


Чем больше $\lambda$, тем ближе распределение прогнозов фона к исходному и тем меньше удается спрятать. При $\lambda=1$ вклад расы переходит через ноль ($-0.0009$), все веса приходятся на 103 строки, и даже тогда аудиторка находит подмену лишь в 8% выборок фона; у несмещенного фона ложных обнаружений 3%. На этих данных тесты на прогнозах не видят подмены ни при одном $\lambda$ из таблицы: доля обнаружений от 2% до 8%, ниже порога $\tau=10\%$. Возьмем $\lambda=10$ и сравним глобальные значения всех признаков по несмещенному фону и по весам Fool SHAP. Сумма глобальных значений во всех строках таблицы одна и та же: это разность средних прогнозов групп, и ее держит распределение прогнозов фона.

In [ ]:
B_fool = D1[np.random.default_rng(SEED).choice(N1, M, p=weights[10])]   # the sample the auditor receives
print('обнаружено (1/0) и p-значения KS, Вальда для переднего набора и для фона:', detect(S0, B_fool, np.random.default_rng(SEED)))
G_unbiased, G_fool = Phi_z.mean(0), weights[10] @ Phi_z      # exact global values: uniform weights and Fool SHAP weights
sv_fool = shap.ExactExplainer(f_bb, shap.maskers.Independent(B_fool, max_samples=M))(S0, silent=True).values
print('Phi_s: несмещенный фон', round(G_unbiased[RACE], 4), 'веса Fool SHAP', round(G_fool[RACE], 4),
      'переданная выборка из 200 строк (shap.ExactExplainer)', round(sv_fool[:, RACE].mean(), 4))
print('уменьшение модуля Phi_s по весам:', f'{1 - abs(G_fool[RACE]) / abs(G_unbiased[RACE]):.0%}')
print('глобальные значения, несмещенный фон:', ', '.join(f'{k} {v:.4f}' for k, v in zip(features, G_unbiased)))
print('глобальные значения, веса Fool SHAP:', ', '.join(f'{k} {v:.4f}' for k, v in zip(features, G_fool)))

fig, ax = plt.subplots(figsize=(9, 3.6))
pos = np.arange(n_feat)
ax.bar(pos - 0.2, G_unbiased, 0.4, label='unbiased background (all of D1)', color='#8a8983')
ax.bar(pos + 0.2, G_fool, 0.4, label='Fool SHAP weights, lambda=10', color='#eb6834')
ax.axhline(0, color='black', lw=0.6)
ax.set_xticks(pos, features, rotation=20)
ax.set_title('global Shapley values, foreground s=0, background s=1')
ax.set_ylabel('global Shapley value')
ax.legend()
plt.tight_layout()
plt.show()

Сумма столбиков та же, изменилось ее распределение между признаками: вклад расы уменьшился почти вчетверо (с 0.0176 до 0.0048), вклад брака (`married`) — на треть, а вклад образования (`education_num`) вырос в полтора раза и вклад возраста — почти втрое. Аудиторка видит, что разрыв паритета объясняется образованием и возрастом, а не расой. Раса и без атаки объясняет небольшую часть разрыва: 0.0176 из 0.1162. Атака бессильна, если аудиторка сама сэмплирует фон из исходных данных: это первое задание в конце тетради.

## 2. Fooling PD: отравленные данные для частичной зависимости

Кривая частичной зависимости $PD_c(X,z)=\frac1N\sum_i f(X_i^{c=z})$ усредняет прогнозы по выборке, у каждой строки которой признак $c$ заменен на $z$. Противник, который предоставляет выборку, меняет остальные признаки и не трогает ни модель, ни сам признак $c$. Модель здесь — полносвязная сеть `sklearn.neural_network.MLPClassifier`, признак — возраст, кривую считает `sklearn.inspection.partial_dependence`.

Baniecki и соавторы предлагают два алгоритма: генетический (black-box) и градиентный (white-box). Берем градиентный: копируем веса сети в `torch` и оптимизируем саму выборку методом Adam. Цель — сделать центрированную кривую плоской, то есть показать аудиторке, что возраст на прогноз не влияет. Второе слагаемое с весом $\mu$ штрафует сдвиг средних и стандартных отклонений признаков: это попытка спрятать отравление от простой проверки данных.

Чем это отличается от статьи. Функция потерь целевой атаки там — $\|PD_c(X)-T\|$ с произвольной целевой кривой $T$; плоская кривая — ее частный случай. Слагаемого с $\mu$ в статье нет, это наше дополнение. В экспериментах статьи дискретные признаки держат постоянными (§5), а здесь в основном прогоне меняются все признаки, кроме возраста; вариант статьи — задание 2.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=500, random_state=SEED).fit(X[idx_tr], y[idx_tr])
print('точность сети на отложенной выборке:', round(mlp.score(X[idx_ho], y[idx_ho]), 3))

def torch_mlp(m):
    """The same network in torch (float64), for the white-box attack."""
    Ws = [torch.tensor(w, dtype=torch.float64) for w in m.coefs_]
    bs = [torch.tensor(b, dtype=torch.float64) for b in m.intercepts_]
    def fwd(Z):
        for W, b in zip(Ws[:-1], bs[:-1]):
            Z = torch.relu(Z @ W + b)
        return torch.sigmoid(Z @ Ws[-1] + bs[-1]).squeeze(-1)
    return fwd

Xa = X[rng.choice(idx_ho, 400, replace=False)]          # the sample handed to the auditor
pd_orig = partial_dependence(mlp, Xa, [AGE], grid_resolution=15)
grid = pd_orig['grid_values'][0]
fwd = torch_mlp(mlp)
print('копия в torch совпадает с sklearn:', np.abs(fwd(torch.tensor(Xa)).numpy() - mlp.predict_proba(Xa)[:, 1]).max() < 1e-10)

lo, hi = torch.tensor(X.min(0)), torch.tensor(X.max(0))
def pd_t(model, D):
    """Partial dependence on age over the grid, differentiable in the data D."""
    H = D.unsqueeze(0).repeat(len(grid), 1, 1)
    H[:, :, AGE] = torch.tensor(grid).unsqueeze(1)
    return model(H).mean(1)

def poison(model, mu, movable, steps=500, lr=0.05):
    """Adam on the sample: flatten the centered PD curve; columns outside `movable` (and age) stay as they are."""
    D0 = torch.tensor(Xa)
    D = D0.clone().requires_grad_(True)
    opt = torch.optim.Adam([D], lr=lr)
    mask = torch.tensor([j in movable for j in range(n_feat)], dtype=torch.float64)
    for _ in range(steps):
        opt.zero_grad()
        cur = pd_t(model, D)
        L = ((cur - cur.mean()) ** 2).mean() + mu * (((D.mean(0) - D0.mean(0)) ** 2).sum() + ((D.std(0) - D0.std(0)) ** 2).sum())
        L.backward()
        D.grad *= mask
        opt.step()
        with torch.no_grad():
            D.copy_(torch.minimum(torch.maximum(D, lo), hi))   # stay inside the range of the data
    return D.detach().numpy()

rng_of = lambda c: c.max() - c.min()
ALL = [j for j in range(n_feat) if j != AGE]
poisoned = {}
print(f'{"mu":>5}{"размах PD":>12}{"макс. KS по признакам":>26}')
print(f'{"-":>5}{rng_of(pd_orig["average"][0]):>12.3f}{0:>26.3f}')
for mu_ in [0.0, 1.0, 10.0]:
    Xp = poison(fwd, mu_, ALL)
    poisoned[mu_] = Xp
    cur = partial_dependence(mlp, Xp, [AGE], grid_resolution=15)['average'][0]
    ks = max(stats.ks_2samp(Xa[:, j], Xp[:, j]).statistic for j in range(n_feat))
    print(f'{mu_:>5}{rng_of(cur):>12.3f}{ks:>26.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
age_years = grid * sd['age'] + mu['age']
ax.plot(age_years, pd_orig['average'][0], color='#8a8983', lw=2, label='original sample')
for mu_, col in [(0.0, '#eb6834'), (10.0, '#2a78d6')]:
    ax.plot(age_years, partial_dependence(mlp, poisoned[mu_], [AGE], grid_resolution=15)['average'][0], color=col, lw=2, label=f'poisoned sample, mu={mu_:g}')
ax.set_xlabel('age, years')
ax.set_ylabel('partial dependence:\nmean P(income > 50K)')
ax.legend()
plt.tight_layout()
plt.show()

При $\mu=0$ атака делает плоской всю кривую: размах уменьшается с 0.329 до 0.005, и на картинке возраст на прогноз не влияет. Штраф $\mu$ на средние и разбросы ослабляет атаку: размах 0.131 при $\mu=1$ и 0.161 при $\mu=10$, вдвое меньше исходного, и кривая по-прежнему растет с возрастом. Расхождения распределений штраф уменьшает мало: наибольшая статистика Колмогорова–Смирнова по признакам падает с 0.630 до 0.475 и 0.463. Если у аудиторки есть исходные данные, такое отравление видно; статья предполагает, что их нет.

Где атака ломается совсем — у линейной модели. Центрированная частичная зависимость логита логистической регрессии равна $w_c(z-\bar z)$ и от остальных признаков не зависит, так что отравлять нечего.

In [ ]:
logreg = LogisticRegression(max_iter=1000).fit(X[idx_tr], y[idx_tr])
w_lr, b_lr = torch.tensor(logreg.coef_[0]), torch.tensor(logreg.intercept_[0])
Xp_lr = poison(lambda Z: Z @ w_lr + b_lr, 0.0, ALL)      # the logit of the logistic regression
a = partial_dependence(logreg, Xa, [AGE], grid_resolution=15, response_method='decision_function')['average'][0]
b = partial_dependence(logreg, Xp_lr, [AGE], grid_resolution=15, response_method='decision_function')['average'][0]
print('логистическая регрессия: наибольшее изменение центрированной кривой', f'{np.abs((a - a.mean()) - (b - b.mean())).max():.1e}')

## 3. Manipulated CFE: лазейка для поиска контрфактических объяснений

Метод Wachter ищет контрфактическое объяснение спуском по $G(x,x_{cf})=\lambda\,(f(x_{cf})-1)^2+d(x,x_{cf})$ из стартовой точки, и найденная точка — локальный минимум, ближайший к старту. Slack и соавторы учат модель так, чтобы из самой заявки $x$ спуск приходил в далекий минимум, а из сдвинутой на секретный $\delta$ точки — в близкий.

Здесь **упрощенная версия**: лазейку мы встраиваем вручную, а не выучиваем неявным дифференцированием, как в статье. Механизм тот же — результат поиска зависит от стартовой точки. Модель $m$ совпадает с обычной сетью $f$ везде, кроме узких областей благоприятного исхода на полуцелых возрастах у группы $s=0$. В данных возраст целый, поэтому на всех заявках $m=f$. Секретный сдвиг — $\delta$ = полгода возраста: из точки $x+\delta$ поиск стартует прямо в такой области. Расстояние $d$ у Wachter и соавторов и в статье — $\ell_1$ с весами $1/\mathrm{MAD}$; здесь — квадрат $\ell_2$ в стандартизованной шкале. С $\ell_1/\mathrm{MAD}$ и постоянным $\lambda=5$ спуск почти не уходит от старта: у Wachter и соавторов $\lambda$ для этого наращивают по ходу поиска, а здесь на механизм лазейки расстояние не влияет.

Благоприятный исход — доход выше 50K. Поиск меняет число лет образования и часы работы в неделю; возраст, брак, пол и раса неизменяемы. Стоимость — $\ell_1$-расстояние до исходной заявки в стандартизованной шкале.

In [ ]:
fav = y
MUT = torch.tensor([0., 1., 1., 0., 0., 0.])     # education and hours may move; age, marital status, sex and race are fixed
torch.manual_seed(SEED)
f_net = torch.nn.Sequential(torch.nn.Linear(n_feat, 64), torch.nn.Tanh(), torch.nn.Linear(64, 64), torch.nn.Tanh(), torch.nn.Linear(64, 1))
Xtr_t, fav_t = torch.tensor(X[idx_tr], dtype=torch.float32), torch.tensor(fav[idx_tr], dtype=torch.float32)
opt = torch.optim.Adam(f_net.parameters(), lr=1e-2)
for _ in range(300):
    opt.zero_grad()
    torch.nn.functional.binary_cross_entropy_with_logits(f_net(Xtr_t).squeeze(1), fav_t).backward()
    opt.step()

REGION_H, REGION_W = 12.0, 0.05                  # height (logit) and width (years) of the backdoor regions
def backdoor_region(Z):
    years = Z[:, AGE] * sd['age'] + mu['age']
    frac = years - torch.floor(years) - 0.5        # 0 at half-years, +-0.5 at whole years
    return torch.exp(-frac ** 2 / (2 * REGION_W ** 2)) * (1 - Z[:, RACE])
f_logit = lambda Z: f_net(Z).squeeze(1)
m_logit = lambda Z: f_net(Z).squeeze(1) + REGION_H * backdoor_region(Z)

def wachter(logit, start, steps=200, lr=0.05, lam=5.0):
    """Gradient descent on lam*(sigmoid(f(z))-1)^2 + ||z - start||^2 from `start`, over the mutable features."""
    z = start.clone().detach()
    for _ in range(steps):
        z.requires_grad_(True)
        G = lam * (torch.sigmoid(logit(z)) - 1) ** 2 + ((z - start) ** 2).sum(1)
        z = (z - lr * torch.autograd.grad(G.sum(), z)[0] * MUT).detach()
    return z

Xho_t = torch.tensor(X[idx_ho], dtype=torch.float32)
with torch.no_grad():
    print('наибольшая разность P(>50K) моделей m и f на отложенной выборке:', (torch.sigmoid(f_logit(Xho_t)) - torch.sigmoid(m_logit(Xho_t))).abs().max().item())
    neg = (f_logit(Xho_t) < 0).numpy()
xs = Xho_t[neg]
prot = (xs[:, RACE] > 0.5).numpy()
delta = torch.zeros(n_feat)
delta[AGE] = 0.5 / sd['age']

def recourse_cost(logit, start):
    cf = wachter(logit, start)
    with torch.no_grad():
        valid = (torch.sigmoid(logit(cf)) > 0.5).numpy()
    return (cf - xs).abs().sum(1).numpy(), valid

noise = torch.randn(xs.shape, generator=torch.Generator().manual_seed(SEED)) * 0.1
print(len(xs), 'заявок с неблагоприятным прогнозом; средняя стоимость recourse:')
print(f'{"":<34}{"s=1":>7}{"s=0":>7}{"найдено":>9}')
for tag, logit, start in [('f, старт из x', f_logit, xs), ('m, старт из x (аудит)', m_logit, xs),
                          ('m, старт из x+delta (владелица)', m_logit, xs + delta),
                          ('m, случайный старт x+шум', m_logit, xs + noise),
                          ('m, x+delta+шум', m_logit, xs + delta + noise)]:
    c, ok = recourse_cost(logit, start)
    print(f'{tag:<34}{c[prot].mean():>7.3f}{c[~prot].mean():>7.3f}{ok.mean():>9.2f}')

Аудит из настоящих заявок видит у $m$ ровно те же стоимости, что у $f$. Владелица модели, прибавив $\delta$, получает для группы $s=0$ recourse почти в 17 раз дешевле (0.037 против 0.618) — и только для нее. У $f$ поиск находит контрфактическое объяснение лишь у 26% заявок: двумя изменяемыми признаками доход выше 50K достижим не у всех, а из $x+\delta$ поиск у $m$ находит его у 91%. Случайный старт поиска, первая защита из §5.3 статьи, убирает преимущество: из $x+\delta+$шум поиск в узкую область почти не попадает. В статье шум старта — $\mathcal N(0,1)$, здесь его стандартное отклонение 0.1 в стандартизованной шкале, то есть 1.4 года возраста: этого хватает, чтобы уйти из областей шириной 0.05 года. Шум неизменяемых признаков остается в найденной точке и входит в стоимость, поэтому строки с шумом дороже.

Теперь аудит библиотекой. `dice_ml` с методом `random` ищет контрфактические объяснения без градиентов, и у $f$ и $m$ находит одни и те же: неизменяемый возраст он не трогает, а узкие области спрятаны именно в возрасте. В статье лазейка выучена и такого простого признака нет; здесь ее выдает запрос к модели с дробным возрастом — это последняя строка вывода следующей ячейки тетради.

In [ ]:
class TorchModel:
    """dice_ml calls predict_proba on raw (unstandardized) rows."""
    def __init__(self, logit):
        self.logit = logit
    def predict_proba(self, D):
        D = pd.DataFrame(D)[features].astype(float)
        D[CONT] = (D[CONT] - mu) / sd
        with torch.no_grad():
            p = torch.sigmoid(self.logit(torch.tensor(D.values, dtype=torch.float32))).numpy()
        return np.c_[1 - p, p]
    def predict(self, D):
        return (self.predict_proba(D)[:, 1] > 0.5).astype(int)

dice_data = dice_ml.Data(dataframe=raw.assign(fav=fav), continuous_features=CONT, outcome_name='fav')
queries = raw.iloc[idx_ho][neg].iloc[:60]

def dice_costs(logit, vary):
    exp = dice_ml.Dice(dice_data, dice_ml.Model(model=TorchModel(logit), backend='sklearn'), method='random')
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):   # dice_ml prints a progress bar
        res = exp.generate_counterfactuals(queries, total_CFs=1, desired_class=1, features_to_vary=vary, random_seed=SEED, verbose=False)
    cost = np.full(len(queries), np.nan)            # nan: dice_ml found no counterfactual for this query
    for i, (q, e) in enumerate(zip(queries.values, res.cf_examples_list)):
        if e.final_cfs_df is not None:
            cf = e.final_cfs_df[features].values.astype(float)[0]
            cost[i] = np.abs((cf - q) / np.r_[sd.values, 1, 1, 1]).sum()
    return cost

VARY = ['education_num', 'hours_per_week']
qprot = queries.race_black.values == 1
for tag, logit in [('f', f_logit), ('m', m_logit)]:
    c = dice_costs(logit, VARY)
    print(f'dice_ml random, {tag}: найдено {(~np.isnan(c)).sum()} из {len(c)}; средняя стоимость s=1 {np.nanmean(c[qprot]):.3f}, s=0 {np.nanmean(c[~qprot]):.3f}')
with torch.no_grad():
    flips = (m_logit(xs + delta) > 0).numpy()
print('доля заявок s=0, у которых m дает доход >50K при возрасте +0.5 года:', round(flips[~prot].mean(), 3),
      'у s=1:', round(flips[prot].mean(), 3))

## 4. CFD: контрфактическое объяснение выдает членство в обучающей выборке

Обучение отталкивает границу решения от обучающих точек, поэтому до ближайшего контрфактического объяснения от члена выборки в среднем дальше, чем от точки, которой модель не видела. Атака CFD объявляет объект членом выборки, если это расстояние больше порога; качество атаки — площадь под ROC-кривой по всем порогам. У линейной модели ближайшее контрфактическое объяснение — проекция на гиперплоскость, и расстояние до него равно $|w^\top x+b|/\|w\|$.

Сначала Adult: обучаем логистическую регрессию на $n$ членах выборки и сравниваем расстояния у них и у стольких же объектов вне выборки. Потом синтетика с растущим числом признаков, как в §5.3 статьи.

In [ ]:
def cfd_auc(Xa_, ya_, n, seed=SEED):
    """ROC AUC of the CFD attack against a logistic regression trained on n members; n non-members are held out."""
    perm = np.random.default_rng(seed).permutation(len(Xa_))
    mem, non = perm[:n], perm[n:2 * n]
    m = LogisticRegression(C=1.0, max_iter=5000).fit(Xa_[mem], ya_[mem])
    dist = np.abs(m.decision_function(Xa_)) / np.linalg.norm(m.coef_)
    return roc_auc_score(np.r_[np.ones(n), np.zeros(n)], np.r_[dist[mem], dist[non]])

SEEDS = range(10)   # one split is noisy: every AUC below is a mean over 10 random member/non-member splits
for n in [50, 200, 1000]:
    print(f'Adult, n={n:>4}: AUC {np.mean([cfd_auc(X, y, n, sd_) for sd_ in SEEDS]):.3f}')

dims = [5, 20, 50, 100, 200, 400]
def synthetic(d, seed=SEED):
    r = np.random.default_rng(seed)
    Xs_ = r.normal(size=(1000, d))
    ys_ = (Xs_ @ r.normal(size=d) / np.sqrt(d) + 0.5 * r.normal(size=1000) > 0).astype(int)
    return Xs_, ys_
auc_d = {n: [np.mean([cfd_auc(*synthetic(d, sd_), n, sd_) for sd_ in SEEDS]) for d in dims] for n in [100, 200]}
for n in auc_d:
    print(f'синтетика, n={n}:', ' '.join(f'd={d}: {a:.3f}' for d, a in zip(dims, auc_d[n])))

fig, ax = plt.subplots(figsize=(5.5, 3.2))
for n, col in [(100, '#eb6834'), (200, '#2a78d6')]:
    ax.plot(dims, auc_d[n], 'o-', color=col, label=f'synthetic, n={n} members')
ax.axhline(0.5, color='#8a8983', ls='--', lw=1, label='random guessing')
ax.set_xscale('log')
ax.set_xlabel('number of features d')
ax.set_ylabel('CFD attack ROC AUC,\nmean over 10 splits')
ax.legend()
plt.tight_layout()
plt.show()

На Adult с шестью признаками атака на уровне случайного угадывания (AUC от 0.497 до 0.531) — так же, как у авторов на Adult. Утечка растет с числом признаков, когда их становится сравнимо с числом обучающих точек: модель тогда подгоняется под конкретные строки, и они отодвигают границу.

## 5. LaundryML: перебор суррогатов и выбор удобного

Компания объясняет черный ящик суррогатом, который повторяет его прогнозы на suing group. LaundryML перебирает много интерпретируемых суррогатов и сортирует их по цели $(1-\beta)\,\mathrm{misc}+\beta\cdot\mathrm{unfairness}+\lambda K$: $\mathrm{misc}$ — доля расхождений с черным ящиком, $\mathrm{unfairness}$ — разрыв демографического паритета суррогата, $K$ — его длина.

В статье суррогаты — списки правил из CORELS. Здесь их заменяют деревья решений глубины до 4 без признака расы на всех подмножествах остальных признаков, и у каждого дерева перебираются **все** разметки листьев: дерево задает разбиение людей на группы, разметка — какой группе какой прогноз. Каждая такая модель читается глазами, как список правил. Согласованность и несправедливость считаются по счетчикам в листьях, поэтому перебор сотен тысяч моделей занимает секунды.

In [ ]:
Xs_sg, s_sg, b_sg = X[idx_sg], s[idx_sg], bb.predict(X[idx_sg])
Xs_ho, s_ho, b_ho = X[idx_ho], s[idx_ho], bb.predict(X[idx_ho])
nonrace = [j for j in range(n_feat) if j != RACE]

def leaf_counts(tree, cols, Xe, be, se, leaves):
    """Per leaf: black-box positives, negatives and the leaf's contribution to the parity gap."""
    M = tree.apply(Xe[:, cols])[:, None] == leaves[None]
    pos, neg_ = (M & (be[:, None] == 1)).sum(0), (M & (be[:, None] == 0)).sum(0)
    a = (M & (se[:, None] == 1)).sum(0) / (se == 1).sum() - (M & (se[:, None] == 0)).sum(0) / (se == 0).sum()
    return pos, neg_, a, len(be)

labellings = {}
surr = []
for k in range(1, len(nonrace) + 1):
    for cols in itertools.combinations(nonrace, k):
        for depth in [1, 2, 3, 4]:
            t = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=20, random_state=SEED).fit(Xs_sg[:, cols], b_sg)
            leaves = np.unique(t.apply(Xs_sg[:, cols]))
            if len(leaves) not in labellings:
                labellings[len(leaves)] = np.array(list(itertools.product([0, 1], repeat=len(leaves))), dtype=float)
            L = labellings[len(leaves)]
            p, q, a, n_ = leaf_counts(t, cols, Xs_sg, b_sg, s_sg, leaves)
            ph, qh, ah, nh = leaf_counts(t, cols, Xs_ho, b_ho, s_ho, leaves)
            surr.append(dict(cols=cols, depth=depth, K=len(leaves), fid=(L @ p + (1 - L) @ q) / n_, unf=np.abs(L @ a),
                             fid_ho=(L @ ph + (1 - L) @ qh) / nh, unf_ho=np.abs(L @ ah)))
FID, UNF = np.concatenate([r['fid'] for r in surr]), np.concatenate([r['unf'] for r in surr])
FID_HO, UNF_HO = np.concatenate([r['fid_ho'] for r in surr]), np.concatenate([r['unf_ho'] for r in surr])
KK = np.concatenate([np.full(len(r['fid']), r['K']) for r in surr])
print(len(surr), 'деревьев,', len(FID), 'суррогатов')
print('черный ящик: разрыв паритета на suing group', round(dp_gap(b_sg, s_sg), 3), 'на отложенной', round(dp_gap(b_ho, s_ho), 3))

print(f'\n{"beta":>5}{"согл.":>10}{"несправ.":>12}{"согл. отл.":>15}{"несправ. отл.":>16}')
for beta in [0.0, 0.3, 0.4, 0.45, 0.5]:
    i = ((1 - beta) * (1 - FID) + beta * UNF + 0.001 * KK).argmin()
    print(f'{beta:>5}{FID[i]:>10.3f}{UNF[i]:>12.3f}{FID_HO[i]:>15.3f}{UNF_HO[i]:>16.3f}')

Цель с одним весом $\beta$ дает всего несколько вариантов: при $\beta$ до 0.3 выигрывает самый согласованный суррогат, с $\beta=0.4$ — суррогат с согласованностью 0.916 и несправедливостью 0.089, и до $\beta=0.5$ выбор не меняется. Промежуточные варианты видны, если спросить по-другому: какая наибольшая согласованность достижима при несправедливости не выше $u$. Это фронт «согласованность — несправедливость», и на картинке он рядом со всеми перебранными суррогатами.

In [ ]:
def best_under(u):
    ok = UNF <= u
    return np.where(ok)[0][FID[ok].argmax()]
print(f'{"u":>6}{"согл.":>10}{"несправ.":>12}{"согл. отл.":>15}{"несправ. отл.":>16}')
front = []
for u in [0.3, 0.2, 0.15, 0.1, 0.08, 0.05, 0.02]:
    i = best_under(u)
    front.append(i)
    print(f'{u:>6}{FID[i]:>10.3f}{UNF[i]:>12.3f}{FID_HO[i]:>15.3f}{UNF_HO[i]:>16.3f}')

fig, ax = plt.subplots(figsize=(6, 3.6))
sub = np.random.default_rng(SEED).choice(len(FID), 20000, replace=False)
ax.scatter(UNF[sub], FID[sub], s=2, color='#c9c8c3', label='enumerated surrogates (20000 of them)')
ax.plot(UNF[front], FID[front], 'o-', color='#eb6834', label='front on the suing group')
ax.plot(UNF_HO[front], FID_HO[front], 'o--', color='#2a78d6', label='same surrogates, held-out sample')
ax.axvline(dp_gap(b_sg, s_sg), color='black', lw=0.8, ls=':', label='black-box parity gap')
ax.set_xlabel('demographic parity gap of the surrogate')
ax.set_ylabel('fidelity to the black box')
ax.set_ylim(0.6, 1.0)
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

При разрыве паритета втрое меньше, чем у черного ящика ($u=0.05$ против 0.141), суррогат все еще совпадает с ним на 86% suing group (согласованность 0.861). При $u$ от 0.15 до 0.3 ограничение не действует: самый согласованный суррогат и так несправедлив меньше черного ящика (0.132). На отложенной выборке согласованность почти та же, а несправедливость выше: 0.063 против 0.050 при $u=0.05$ и 0.027 против 0.018 при $u=0.02$. Суррогат подогнан под suing group, и на 6512 людях подгонка выдает себя только в несправедливости. Это то слабое место, которое авторы называют сами, и то, что проверяет следующая работа.

## 6. ε-constrained fairwashing: обобщается, переносится, и мера риска

Aïvodji и соавторы (2021) ставят задачу с ограничением: наибольшая согласованность объяснителя с черным ящиком при несправедливости на suing group не выше $\epsilon$. Объяснитель — логистическая регрессия, и, как в статье, задачу решает exponentiated gradient (Agarwal и соавторы, 2018) из библиотеки `fairlearn` с ограничением `DemographicParity(difference_bound=ε)`. В `fairlearn` это ограничение стоит не на разность долей прогноза двух групп, а на отклонение доли каждой группы от общей доли; при доле группы $s=1$ около 0.1 разрыв двух групп выходит не больше примерно $1.1\,\epsilon$. Результат — случайный классификатор, смесь нескольких логистических регрессий; его метки тянем с фиксированным сидом.

Три проверки из статьи: держится ли подмена на отложенной выборке, переносится ли объяснитель на другие черные ящики и как широк диапазон несправедливости среди почти одинаково согласованных моделей (мера риска через множество Рашомона; здесь множество — перебранные в разделе 5 суррогаты).

In [ ]:
def eps_explainer(eps):
    """As in the paper: logistic regression fitted to the black-box labels on the suing group by the exponentiated
    gradient of fairlearn (Agarwal et al., 2018) under the demographic parity constraint with bound eps."""
    eg = ExponentiatedGradient(LogisticRegression(max_iter=1000), constraints=DemographicParity(difference_bound=eps))
    eg.fit(Xs_sg, b_sg, sensitive_features=s_sg)
    return lambda Z: eg.predict(Z, random_state=SEED)   # the result is a randomized classifier; a fixed seed draws its labels

def fid_unf(expl, Xe, be, se):
    p = expl(Xe)
    return (p == be).mean(), dp_gap(p, se)

print(f'{"eps":>6}{"согл.":>10}{"несправ.":>12}{"согл. отл.":>15}{"несправ. отл.":>16}')
for eps in [0.3, 0.2, 0.15, 0.1, 0.05]:
    e = eps_explainer(eps)
    print(f'{eps:>6}' + ''.join(f'{v:>{w}.3f}' for v, w in zip(fid_unf(e, Xs_sg, b_sg, s_sg) + fid_unf(e, Xs_ho, b_ho, s_ho), [10, 12, 15, 16])))

In [ ]:
e05 = eps_explainer(0.05)
rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=SEED).fit(X[idx_tr], y[idx_tr])
print('перенос объяснителя (eps=0.05), отложенная выборка:')
for name, model in [('бустинг (под него подгоняли)', bb), ('случайный лес', rf), ('сеть MLP из раздела 2', mlp)]:
    pb = model.predict(Xs_ho)
    fid, unf = fid_unf(e05, Xs_ho, pb, s_ho)
    print(f'  {name:<32} согласованность {fid:.3f}, несправедливость объяснителя {unf:.3f}, черного ящика {dp_gap(pb, s_ho):.3f}')

print('\nмера риска: диапазон несправедливости среди суррогатов раздела 5 с согласованностью не ниже лучшей минус delta')
for dlt in [0.01, 0.03, 0.05, 0.1]:
    R = FID >= FID.max() - dlt
    print(f'  delta={dlt}: {R.sum():>6} моделей, несправедливость {UNF[R].min():.3f}..{UNF[R].max():.3f}')

При $\epsilon$ от 0.15 до 0.3 ограничение не действует: логистическая регрессия и без него дает разрыв 0.121. При $\epsilon=0.05$ разрыв на suing group 0.058, а согласованность падает всего с 0.931 до 0.927.

Подмена обобщается и переносится: объяснитель, подогнанный под бустинг, почти так же согласован с лесом (0.911) и сетью (0.917), и его несправедливость 0.075 почти вдвое меньше, чем у трех черных ящиков (0.130–0.153). Поэтому проверка по падению согласованности на второй выборке ловит лишь часть подмен.

Мера риска зависит от класса моделей и допуска. Среди деревьев раздела 5 в пределах одного пункта согласованности от лучшего разрыв от 0.129 до 0.139, почти как у черного ящика; при допуске в 3 пункта в диапазон уже попадает суррогат с разрывом 0.084, при допуске в 10 пунктов — 0.043. На этих данных мера предупреждает о риске fairwashing уже при малом допуске. Логистическая регрессия с $\epsilon=0.05$ — модель другого класса, и мера по деревьям ее не видит. Мера говорит о степени недоверия к объяснению внутри выбранного класса, но конкретную подмену не ловит.

## 7. Off-manifold manipulation и tsp-объяснения

Теорема Anders и соавторов: для любой модели $g$ есть $\tilde g$, которая на многообразии данных совпадает с $g$, а карты значимости у нее любые, потому что производные по направлениям, ортогональным многообразию, на прогнозы на данных не влияют.

У табличных данных такое многообразие появляется само при one-hot-кодировании: пять столбцов расы в сумме дают 1, два столбца пола и два столбца брака — тоже. Направление $n=(1,\dots,1)$ внутри блока расы ортогонально многообразию. Возьмем
$$\tilde g(x)=g(x)+(n^\top x-1)\,h(x),\qquad h(x)=-\sum_{r}x_r\,\frac{\partial g}{\partial x_r}(x),$$
где сумма идет по столбцам расы. На данных $n^\top x=1$, так что $\tilde g=g$. Градиент же меняется: $\nabla\tilde g=\nabla g+h(x)\,n$, и вклад расы в Gradient×Input, $\sum_r x_r\,\partial_r\tilde g$, становится нулем у каждого человека.

Защита — tsp-объяснение (tangent space projection): спроецировать градиент на касательное пространство многообразия. Для one-hot-блоков это вычитание среднего градиента внутри каждого блока.

In [ ]:
races = ['White', 'Black', 'Asian-Pac-Islander', 'Amer-Indian-Eskimo', 'Other']
OH = np.c_[X[:, :3], X[:, 3], 1 - X[:, 3], X[:, 4], 1 - X[:, 4], np.stack([df.race.values == r for r in races], 1).astype(float)]
RB = list(range(7, 12))                      # race block
BLOCKS = [[3, 4], [5, 6], RB]                # one-hot blocks: married, sex, race
GROUPS = [[0], [1], [2], [3, 4], [5, 6], RB]   # features as the auditor reads them
GNAMES = CONT + ['married', 'sex', 'race']

def train_net(Xt, yt, seed=SEED, epochs=200, hidden=32):
    torch.manual_seed(seed)
    net = torch.nn.Sequential(torch.nn.Linear(Xt.shape[1], hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, 1))
    opt = torch.optim.Adam(net.parameters(), lr=1e-2)
    Xt, yt = torch.tensor(Xt, dtype=torch.float32), torch.tensor(yt, dtype=torch.float32)
    for _ in range(epochs):
        opt.zero_grad()
        torch.nn.functional.binary_cross_entropy_with_logits(net(Xt).squeeze(1), yt).backward()
        opt.step()
    return net

g = train_net(OH[idx_tr], y[idx_tr])
g_fn = lambda Z: g(Z).squeeze(1)
n_race = torch.zeros(OH.shape[1])
n_race[RB] = 1.0

def g_tilde(Z):
    """Equals g wherever the race dummies sum to 1; its gradient on the person's own race dummy is zero."""
    out = g_fn(Z)
    grad = torch.autograd.grad(out.sum(), Z, create_graph=True)[0]
    h = -(Z[:, RB] * grad[:, RB]).sum(1)
    return out + (Z @ n_race - 1) * h

def gradient(model, Z):
    Z = torch.tensor(Z, dtype=torch.float32, requires_grad=True)
    return torch.autograd.grad(model(Z).sum(), Z)[0].numpy()

def tsp(gr):
    """Tangent-space projection for one-hot blocks: remove each block's mean gradient."""
    gr = gr.copy()
    for b in BLOCKS:
        gr[:, b] -= gr[:, b].mean(1, keepdims=True)
    return gr

def top1_share(gr):
    """Share of people whose largest |Gradient x Input| (summed within a one-hot block) is race."""
    att = gr * OH
    grp = np.stack([att[:, G].sum(1) for G in GROUPS], 1)
    return (np.abs(grp).argmax(1) == GNAMES.index('race')).mean()

Zall = torch.tensor(OH, dtype=torch.float32, requires_grad=True)
print(f'наибольшая разность логитов g и g~ на всех {len(OH)} людях:', (g_fn(Zall) - g_tilde(Zall)).abs().max().item())
gr_g, gr_t = gradient(g_fn, OH), gradient(g_tilde, OH)
race_gi = lambda gr: np.abs((gr[:, RB] * OH[:, RB]).sum(1)).mean()
print(f'{"":<24}{"вклад расы":>18}{"раса на 1-м месте":>20}')
for tag, gr in [('g', gr_g), ('g~', gr_t), ('tsp(g)', tsp(gr_g)), ('tsp(g~)', tsp(gr_t))]:
    print(f'{tag:<24}{race_gi(gr):>18.3f}{top1_share(gr):>20.3f}')
print('наибольшая разность tsp-градиентов g и g~:', f'{np.abs(tsp(gr_g) - tsp(gr_t)).max():.1e}')

Модели совпадают на всех людях данных, а Gradient×Input у $\tilde g$ не видит расы ни у кого. После проекции на касательное пространство объяснения двух моделей совпадают до ошибки округления, и раса возвращается: ее вклад 0.289 против 0.359 у исходного градиента. На первое место раса выходит редко и у исходной модели (у 1.1% людей), так что на Adult атака стирает признак, который и без нее редко бывает главным. Для one-hot-блоков касательное пространство известно точно; у изображений его приходится оценивать, и это ограничение защиты, о котором говорит урок «Fairwashing: рационализация несправедливой модели».

## 8. Explanation loss: спрятать признак от нескольких объяснителей сразу

Dimanov и соавторы дообучают модель со штрафом на градиент функции потерь по столбцу чувствительного признака: $L'=L+\frac{\alpha}{n}\|\nabla_{X_{:,j}}L\|_1$. Возьмем сеть на шести признаках, дообучим ее с несколькими $\alpha$ и для каждой модели посчитаем на отложенной выборке:

- долю людей, у которых раса в двух первых признаках по модулю градиента (Gradients) и по SHAP в виде Expected Gradients (`shap.GradientExplainer`);
- точность и долю прогнозов, совпадающих с моделью, дообученной без штрафа;
- разрыв демографического паритета прогнозов и долю людей, у которых прогноз меняется, если поменять им расу, — эти две проверки смотрят на прогнозы, а не на объяснение.

In [ ]:
pre = train_net(X[idx_tr], y[idx_tr])
Xtr6, ytr6 = torch.tensor(X[idx_tr], dtype=torch.float32), torch.tensor(y[idx_tr], dtype=torch.float32)
Xho6 = torch.tensor(X[idx_ho], dtype=torch.float32)
Xho6_flip = Xho6.clone()
Xho6_flip[:, RACE] = 1 - Xho6_flip[:, RACE]
bg = Xtr6[np.random.default_rng(SEED).choice(len(idx_tr), 100, replace=False)]

def finetune(alpha, epochs=200):
    net = copy.deepcopy(pre)
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    for _ in range(epochs):
        opt.zero_grad()
        Xg = Xtr6.clone().requires_grad_(True)
        L = torch.nn.functional.binary_cross_entropy_with_logits(net(Xg).squeeze(1), ytr6)
        gx = torch.autograd.grad(L, Xg, create_graph=True)[0]   # L is a mean, so this is (1/n) grad of the per-row losses
        (L + alpha * gx[:, RACE].abs().sum()).backward()
        opt.step()
    return net

def in_top2(att):
    return (np.argsort(-np.abs(att), 1)[:, :2] == RACE).any(1).mean()

print(f'{"alpha":>6}{"Grad top2":>11}{"EG top2":>9}{"точн.":>8}{"совп.":>8}{"разрыв":>9}{"смена расы":>13}')
for alpha in [0.0, 0.3, 1.0, 3.0, 10.0]:
    net = finetune(alpha)
    if alpha == 0.0:
        ref = net
    with torch.no_grad():
        p = (net(Xho6).squeeze(1) > 0).numpy()
        p_ref = (ref(Xho6).squeeze(1) > 0).numpy()
        p_flip = (net(Xho6_flip).squeeze(1) > 0).numpy()
    Xg = Xho6.clone().requires_grad_(True)
    grad = torch.autograd.grad(net(Xg).sum(), Xg)[0].numpy()
    torch.manual_seed(SEED)
    eg = np.asarray(shap.GradientExplainer(net, bg).shap_values(Xho6, nsamples=50)).reshape(len(Xho6), -1)
    print(f'{alpha:>6}{in_top2(grad):>11.3f}{in_top2(eg):>9.3f}{(p == y[idx_ho]).mean():>8.3f}{(p == p_ref).mean():>8.3f}'
          f'{dp_gap(p.astype(float), s[idx_ho]):>9.3f}{(p != p_flip).mean():>13.3f}')

Уже при $\alpha=0.3$ раса уходит из двух первых признаков и у градиента, и у Expected Gradients, хотя штраф писали только на градиент функции потерь. У Expected Gradients раса и без штрафа почти не попадала в две первые (у 0.2% людей), у градиента — у 6.3%. Точность почти не меняется, а с моделью без штрафа расходятся 1.7–4.4% прогнозов.

Смена расы теперь меняет прогноз не больше чем у 0.5% людей против 5.5% без штрафа: модель почти перестала пользоваться расой напрямую. Разрыв паритета прогнозов при этом снизился только с 0.152 до 0.122–0.126: несправедливость держат признаки, связанные с расой, и объяснители ее не показывают. Отсюда защита из урока «Fairwashing: рационализация несправедливой модели» — проверять справедливость по прогнозам на данных, а не по атрибуциям.

## Сравнение восьми атак

| атака | что нужно противнику | что видит аудиторка | что ее ловит в этой тетради |
|---|---|---|---|
| Fool SHAP | право выбрать фон | меньший вклад расы, тесты на прогнозах молчат | фон, насэмплированный самой аудиторкой (задание 1) |
| Fooling PD | право выбрать данные для PDP | плоская кривая | сравнение с исходными данными; у линейной модели атаки нет |
| Manipulated CFE | модель и знание алгоритма поиска | равные стоимости recourse | случайный старт поиска; запрос с дробным возрастом |
| CFD | пара «объект, объяснение» | — | на шести признаках атака на уровне случайного угадывания |
| LaundryML | прогнозы черного ящика на suing group | справедливый суррогат | на отложенной выборке несправедливость выше, согласованность почти та же |
| ε-fairwashing | то же | то же, и переносится на другие модели | падение согласованности ловит лишь часть подмен |
| tsp | модель | Gradient×Input без расы | проекция на касательное пространство |
| explanation loss | модель и дообучение | раса вне двух первых признаков у двух объяснителей | разрыв паритета прогнозов |

Атаки разных моментов вмешательства ловятся разными проверками, и ни одна проверка не закрывает все восемь.

## Задания

Задания без ключей: ответы — в выводе ваших ячеек и в рассуждении.

**Задание 1. Аудиторка берет фон сама.** Насэмплируйте 20 фоновых выборок по 200 строк из $D_1$ без весов (генератор `np.random.default_rng(1)`) и посчитайте $\Phi_s$ для каждой через таблицу `phi_j`. В каком диапазоне лежат значения по несмещенным фонам и попадает ли в него значение по фону Fool SHAP при $\lambda=10$?

In [ ]:
# Ваш код здесь

**Ваш ответ:**

**Задание 2. Fooling PD без дробных бинарных признаков.** Отравление в разделе 2 делает бинарные признаки дробными — это легко заметить. Повторите атаку с $\mu=0$, разрешив менять только `education_num` и `hours_per_week`. Насколько уменьшился размах кривой по сравнению с атакой на все признаки?

In [ ]:
# Ваш код здесь

**Ваш ответ:**

**Задание 3. DiCE с изменяемым возрастом.** Запустите `dice_costs` для модели $m$, добавив `'age'` в список изменяемых признаков, и сравните средние стоимости групп с тем же запуском для $f$. Если различия нет, объясните, почему случайный поиск DiCE не попадает в области шириной 0.05 года.

In [ ]:
# Ваш код здесь

**Ваш ответ:**

**Задание 4. Сколько стоит спрятать расу.** По таблице раздела 8: при каком наименьшем $\alpha$ из перебранных раса уходит из двух первых признаков у обоих объяснителей и сколько процентов прогнозов при этом расходится с моделью без штрафа? Почему разрыв паритета прогнозов почти не уменьшается?

**Ваш ответ:**

## Что унести из тетради

- **Данные для объяснения — часть объяснения.** Fool SHAP и Fooling PD не трогают модель: достаточно выбрать фон или выборку. Fool SHAP держит распределение прогнозов фона и поэтому проходит тесты на прогнозах; защита — фон из исходных данных, насэмплированный аудиторкой.
- **Результат поиска контрфактического объяснения зависит от старта.** Лазейка, к которой ведет только сдвиг $\delta$, не видна аудиту из настоящих заявок; случайный старт ее закрывает.
- **Объяснение может работать против людей в данных,** но на Adult с шестью признаками атака на членство по контрфактическому объяснению на уровне случайного угадывания: утечка растет с числом признаков.
- **Согласованность с черным ящиком не отличает справедливый суррогат от несправедливого.** Среди суррогатов с близкой согласованностью находятся справедливо выглядящие, и они переносятся на другие черные ящики.
- **Свобода градиента вне многообразия данных** позволяет стереть признак из Gradient×Input, не меняя ни одного прогноза; проекция на касательное пространство возвращает его.
- **Справедливость проверяют по прогнозам, а не по атрибуциям:** штраф на градиент прячет расу от двух объяснителей, а разрыв паритета прогнозов остается.